# Abu Dhabi PM10 Forecasting — Random Forest Regression

## Modelling

This notebook trains a Random Forest model to predict **tomorrow's PM10 daily average** at each monitoring station. It uses the shared `model_table.csv` and follows the project's predefined chronological split. The notebook covers model training, validation-based tuning, interpretation, and final testing; data exploration and preparation are handled separately.

## Step 1 — Import libraries and define project settings

Import the libraries needed for Random Forest regression, evaluation, visualisation, and saving the final model. Set one random seed so the model experiments are reproducible.

In [ ]:
# Step 1: Import libraries and define project settings

from pathlib import Path
import itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler

RANDOM_STATE = 42
TARGET = "pm10_tomorrow"
SPLIT_COL = "split"
OUTPUT_DIR = Path("/content/rf_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project settings defined.")

### Step 1 — Results

The notebook is set up for a reproducible regression experiment. All model outputs will be saved in `rf_results` so the final model, metrics, and predictions can be included in the project report.

## Step 2 — Load the shared model table

Load `model_table.csv`. In Colab, the upload prompt appears only when the file is not already available in the working directory. We use the prepared modelling table directly; this notebook does not rebuild or modify it.

In [ ]:
# Step 2: Load the shared modelling dataset

DATA_FILE = Path("/content/model_table.csv")

if not DATA_FILE.exists():
    from google.colab import files
    print("Upload the shared model_table.csv file:")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_names:
        raise FileNotFoundError("No CSV file was uploaded. Please upload model_table.csv.")
    DATA_FILE = Path("/content") / csv_names[0]

data = pd.read_csv(DATA_FILE, parse_dates=["day"])

print(f"Loaded: {DATA_FILE.name}")
print(f"Rows: {len(data):,} | Columns: {len(data.columns)}")
print("Split counts:")
print(data[SPLIT_COL].value_counts().reindex(["train", "validation", "test"]))

### Step 2 — Results

The shared table is loaded with its original `split` labels. These labels define the training, validation, and test periods and must remain unchanged so our results are comparable with the other project models.

## Step 3 — Define the prediction inputs and target

The target is `pm10_tomorrow`. The predictors are the seven PM10 measurements from six days ago through today, plus today's NO2 and SO2 measurements. We exclude station identifiers, station names, dates, and the split label because they are metadata rather than pollutant measurements. This keeps the model input focused on the shared environmental features.

In [ ]:
# Step 3: Select model features and target

FEATURES = [
    "pm10_6d_ago", "pm10_5d_ago", "pm10_4d_ago",
    "pm10_3d_ago", "pm10_2d_ago", "pm10_1d_ago",
    "pm10_today", "no2_today", "so2_today"
]

required_columns = FEATURES + [TARGET, SPLIT_COL]
missing_columns = [col for col in required_columns if col not in data.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if data[required_columns].isna().any().any():
    raise ValueError("Missing values found in model features, target, or split column.")

X_train = data.loc[data[SPLIT_COL] == "train", FEATURES].copy()
y_train = data.loc[data[SPLIT_COL] == "train", TARGET].copy()

X_val = data.loc[data[SPLIT_COL] == "validation", FEATURES].copy()
y_val = data.loc[data[SPLIT_COL] == "validation", TARGET].copy()

X_test = data.loc[data[SPLIT_COL] == "test", FEATURES].copy()
y_test = data.loc[data[SPLIT_COL] == "test", TARGET].copy()

if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise ValueError("One of the expected data splits is empty. Check the split column.")

print("Predictors:", FEATURES)
print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_val):,}")
print(f"Test rows: {len(X_test):,}")

### Step 3 — Results

The model uses **nine pollutant features** to predict tomorrow's PM10. The original chronological split is preserved. The test set is held aside and will not be used for training or hyperparameter selection.

## Step 4 — Define evaluation metrics and a persistence benchmark

We evaluate regression performance using MAE, RMSE, and R². As a simple forecasting benchmark, we also predict that tomorrow's PM10 will equal today's PM10 (`pm10_today`). Random Forest should be compared with this persistence baseline, not judged by its score alone.

In [ ]:
# Step 4: Define regression metrics and the persistence benchmark

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def show_metrics(name, y_true, y_pred):
    scores = regression_metrics(y_true, y_pred)
    print(f"{name}: MAE={scores['MAE']:.3f} | RMSE={scores['RMSE']:.3f} | R2={scores['R2']:.3f}")
    return scores

# Today's PM10 is used as the prediction for tomorrow.
persistence_val_pred = data.loc[data[SPLIT_COL] == "validation", "pm10_today"]
persistence_val_scores = show_metrics("Persistence benchmark — validation", y_val, persistence_val_pred)

### Step 4 — Results

The persistence score is our reference point. MAE and RMSE are measured in the same units as PM10; lower values are better. R² measures how much variation is explained by the predictions; higher values are better.

## Step 5 — Train a baseline Random Forest

Train a first Random Forest Regressor using the training period only. We begin with a straightforward configuration before tuning. The validation set is used to assess how well this baseline generalises to a later period.

In [ ]:
# Step 5: Train and evaluate a baseline Random Forest

rf_baseline = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_baseline.fit(X_train, y_train)

baseline_train_pred = rf_baseline.predict(X_train)
baseline_val_pred = rf_baseline.predict(X_val)

baseline_train_scores = show_metrics("Baseline Random Forest — train", y_train, baseline_train_pred)
baseline_val_scores = show_metrics("Baseline Random Forest — validation", y_val, baseline_val_pred)

baseline_comparison = pd.DataFrame([
    {"Model": "Persistence", **persistence_val_scores},
    {"Model": "Baseline Random Forest", **baseline_val_scores},
])
display(baseline_comparison.round(3))

### Step 5 — Results

Compare the Random Forest validation scores with the persistence benchmark. A lower validation RMSE indicates a better average balance of prediction errors. Also compare training and validation scores: a much stronger training result may indicate overfitting.

## Step 6 — Tune Random Forest hyperparameters using validation data

Test a controlled set of Random Forest settings. Each candidate is fitted on the training set and scored on the validation set; the candidate with the lowest validation RMSE is selected. The test set is not involved in this search.

In [ ]:
# Step 6: Randomized hyperparameter search

param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt"],
}

candidate_params = list(ParameterSampler(
    param_distributions,
    n_iter=24,
    random_state=RANDOM_STATE
))

search_results = []
best_model = None
best_params = None
best_val_rmse = np.inf

for run_number, params in enumerate(candidate_params, start=1):
    candidate = RandomForestRegressor(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

    search_results.append({"run": run_number, **params, "validation_RMSE": val_rmse})

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_params = params
        best_model = candidate

search_results_df = pd.DataFrame(search_results).sort_values("validation_RMSE").reset_index(drop=True)
print(f"Candidates evaluated: {len(candidate_params)}")
print(f"Best validation RMSE: {best_val_rmse:.3f}")
print("Best hyperparameters:")
print(best_params)
display(search_results_df.head(10).round(3))

search_results_df.to_csv(OUTPUT_DIR / "rf_hyperparameter_search.csv", index=False)

### Step 6 — Results

The table ranks the tested configurations by validation RMSE. The best settings are selected using validation performance only. This gives us a tuned model without using the final test period to make modelling decisions.

## Step 7 — Compare the baseline and tuned models

Evaluate the selected configuration on the training and validation periods. Compare it with the baseline Random Forest and the persistence benchmark to see whether tuning improved validation performance and whether the model adds value beyond today's PM10 measurement.

In [ ]:
# Step 7: Compare baseline, tuned Random Forest, and persistence

tuned_train_pred = best_model.predict(X_train)
tuned_val_pred = best_model.predict(X_val)

tuned_train_scores = regression_metrics(y_train, tuned_train_pred)
tuned_val_scores = regression_metrics(y_val, tuned_val_pred)

model_comparison = pd.DataFrame([
    {"Model": "Persistence", **persistence_val_scores},
    {"Model": "Baseline Random Forest", **baseline_val_scores},
    {"Model": "Tuned Random Forest", **tuned_val_scores},
])
display(model_comparison.round(3))

improvement = (persistence_val_scores["RMSE"] - tuned_val_scores["RMSE"]) / persistence_val_scores["RMSE"] * 100
print(f"Tuned Random Forest validation RMSE improvement over persistence: {improvement:.2f}%")

### Step 7 — Results

A positive RMSE improvement means the tuned Random Forest made smaller validation errors than the persistence benchmark. If the improvement is negative, the simpler persistence forecast performed better. The validation set is our basis for choosing the final model configuration.

## Step 8 — Interpret the tuned model with permutation importance

Permutation importance estimates how much validation performance changes when one feature is shuffled. A larger increase in RMSE means the model relied more on that feature for its validation predictions. This is a model interpretation tool, not proof that a feature causes PM10 changes.

In [ ]:
# Step 8: Calculate and plot validation permutation importance

perm = permutation_importance(
    best_model,
    X_val,
    y_val,
    n_repeats=15,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=True)

display(importance_df.sort_values("importance_mean", ascending=False).round(4))

plt.figure(figsize=(9, 5))
plt.barh(importance_df["feature"], importance_df["importance_mean"],
         xerr=importance_df["importance_std"])
plt.xlabel("Increase in validation RMSE after shuffling")
plt.ylabel("Feature")
plt.title("Random Forest — permutation feature importance")
plt.tight_layout()
plt.show()

importance_df.to_csv(OUTPUT_DIR / "rf_permutation_importance.csv", index=False)

### Step 8 — Results

Features with larger positive importance values contribute more to validation accuracy in this fitted model. Very small or negative values suggest that shuffling the feature did not harm performance. Interpret the ranking as model-specific evidence, not as a causal explanation.

## Step 9 — Examine validation predictions

Compare actual and predicted PM10 values on the validation period. The scatter plot shows whether predictions follow the ideal 45-degree line; the residual plot helps reveal systematic overprediction or underprediction. These are model diagnostics, not general exploratory data analysis.

In [ ]:
# Step 9: Validation prediction diagnostics

val_predictions = pd.DataFrame({
    "actual_pm10_tomorrow": y_val.to_numpy(),
    "predicted_pm10_tomorrow": tuned_val_pred,
})
val_predictions["residual"] = (
    val_predictions["actual_pm10_tomorrow"] - val_predictions["predicted_pm10_tomorrow"]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(val_predictions["actual_pm10_tomorrow"],
                val_predictions["predicted_pm10_tomorrow"], alpha=0.6)
lims = [
    min(val_predictions["actual_pm10_tomorrow"].min(), val_predictions["predicted_pm10_tomorrow"].min()),
    max(val_predictions["actual_pm10_tomorrow"].max(), val_predictions["predicted_pm10_tomorrow"].max())
]
axes[0].plot(lims, lims, "--")
axes[0].set_xlabel("Actual PM10 tomorrow")
axes[0].set_ylabel("Predicted PM10 tomorrow")
axes[0].set_title("Validation: actual vs predicted")

axes[1].scatter(val_predictions["predicted_pm10_tomorrow"], val_predictions["residual"], alpha=0.6)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted PM10 tomorrow")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Validation: residuals")

plt.tight_layout()
plt.show()

print(f"Mean validation residual: {val_predictions['residual'].mean():.3f}")
print(f"Validation residual standard deviation: {val_predictions['residual'].std():.3f}")

### Step 9 — Results

Points close to the diagonal indicate accurate predictions. Residuals centered around zero suggest limited overall bias; visible patterns or a widening spread may indicate errors that depend on the predicted PM10 level. Use the plots to describe strengths and limitations rather than claiming perfect prediction.

## Step 10 — Fit the final model and evaluate on the test set

After choosing the hyperparameters, refit the model using the combined training and validation data. Then evaluate it once on the untouched test period. Also score the persistence benchmark on the same test rows to measure whether Random Forest improves on the simple last-observation forecast.

**Do not change features or hyperparameters based on these test results.**

In [ ]:
# Step 10: Train final model on train + validation, then test once

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

final_model = RandomForestRegressor(
    **best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
final_model.fit(X_train_val, y_train_val)

# Final test predictions
final_test_pred = final_model.predict(X_test)
persistence_test_pred = data.loc[data[SPLIT_COL] == "test", "pm10_today"]

final_test_scores = regression_metrics(y_test, final_test_pred)
persistence_test_scores = regression_metrics(y_test, persistence_test_pred)

test_comparison = pd.DataFrame([
    {"Model": "Persistence", **persistence_test_scores},
    {"Model": "Final Random Forest", **final_test_scores},
])
print("Final test-set results:")
display(test_comparison.round(3))

test_improvement = (
    (persistence_test_scores["RMSE"] - final_test_scores["RMSE"])
    / persistence_test_scores["RMSE"] * 100
)
print(f"Test RMSE improvement over persistence: {test_improvement:.2f}%")

### Step 10 — Results

These are the final reported test metrics. Compare the Random Forest with persistence using MAE and RMSE, and report R² as a complementary measure. The test set represents later dates, so its performance is the best estimate here of how the chosen model handles unseen future observations from the prepared dataset.

## Step 11 — Save the model and final outputs

Save the final fitted model, the test metrics, and row-level test predictions. These files can be used in the project report and shared repository. The saved prediction file includes the original station and date information for traceability, but these metadata columns were not model inputs.

In [ ]:
# Step 11: Save model, test metrics, and test predictions

joblib.dump(final_model, OUTPUT_DIR / "random_forest_pm10.joblib")

test_comparison.to_csv(OUTPUT_DIR / "rf_test_metrics.csv", index=False)

meta_test = data.loc[data[SPLIT_COL] == "test", ["station_id", "station", "day"]].reset_index(drop=True)
test_predictions = meta_test.copy()
test_predictions["actual_pm10_tomorrow"] = y_test.reset_index(drop=True)
test_predictions["rf_predicted_pm10_tomorrow"] = final_test_pred
test_predictions["persistence_predicted_pm10_tomorrow"] = persistence_test_pred.reset_index(drop=True)
test_predictions["rf_residual"] = (
    test_predictions["actual_pm10_tomorrow"] - test_predictions["rf_predicted_pm10_tomorrow"]
)
test_predictions.to_csv(OUTPUT_DIR / "rf_test_predictions.csv", index=False)

print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

### Step 11 — Results

The final model and its test outputs are saved in `rf_results`. Keep the saved model, metrics, predictions, and hyperparameter search table together so the work is reproducible and easy to compare with the other project models.

## Modelling summary

The modelling workflow is complete. In the presentation, focus on the selected hyperparameters, validation comparison against persistence, the most important features, and the final test metrics. Be clear about whether Random Forest actually improves on the persistence benchmark; a more complex model is not automatically a better forecasting model.